# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Unit of analysis

**One row represents one web/search page observation for a specific time period.**

The page-level observation contains search-performance and content-related information that can be used to identify pages that may need content review or refresh.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
import pandas as pd

url = "https://github.com/syedamominapak-coder/flyrankai_ml_internship/raw/refs/heads/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Dataset loaded successfully!
Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [3]:
# Check the dataset columns first
print(df.columns.tolist())


['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [4]:
# Query 1: Verify the grain
# Check whether each content_id appears only once

total_rows = len(df)
unique_content_ids = df["content_id"].nunique()

print("Total rows:", total_rows)
print("Unique content_ids:", unique_content_ids)
print("Duplicate content_id rows:", total_rows - unique_content_ids)

Total rows: 30000
Unique content_ids: 30000
Duplicate content_id rows: 0


In [5]:
# Query 2: Check for date/time fields

date_like_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in ["date", "month", "week", "time", "period"])
]

print("Date/time fields found:")
print(date_like_cols)

Date/time fields found:
['days_since_last_update']


In [6]:
# Query 2: Verify row count and available observation fields

print("Total rows in Content Refresh lane:", len(df))

print("\n90-day performance fields:")
print([
    "impressions_90d",
    "clicks_90d",
    "sessions_90d"
])

print("\nRecent 30-day performance fields:")
print([
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
])

Total rows in Content Refresh lane: 30000

90-day performance fields:
['impressions_90d', 'clicks_90d', 'sessions_90d']

Recent 30-day performance fields:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']


In [7]:
# Query 3: Verify availability using IS TRUE

available = df["impressions_90d"].notna()

print("Rows where impressions_90d IS TRUE:", available.sum())
print("Total rows:", len(df))
print("Rows available:", available.sum())
print("Rows missing:", (~available).sum())

Rows where impressions_90d IS TRUE: 30000
Total rows: 30000
Rows available: 30000
Rows missing: 0


## 4. Five features

For the Content Refresh / Search Performance lane, I will use at most five features that are available at the decision moment.

The features are selected to describe search demand, recent performance, content age, and content size without using the future outcome label.


In [8]:
feature_1 = df[["search_volume"]].copy()

feature_1.head()

,search_volume
0,10.0
1,90.0
2,0.0
3,10.0
4,0.0


**Feature 1: `search_volume`**

**Available when?** Knowable at the decision moment because search volume describes current search demand for the page's topic and does not depend on the future outcome label.


In [9]:
feature_2 = df[["competition"]].copy()

feature_2.head()

,competition
0,0.67
1,0.01
2,0.00
3,0.00
4,0.00


**Feature 2: `competition`**

**Available when?** Knowable at the decision moment because competition describes the current search environment for the page's topic and does not use the future outcome label.


In [10]:
feature_3 = df[["content_age_days"]].copy()

feature_3.head()

,content_age_days
0,187
1,445
2,141
3,463
4,263


**Feature 3: `content_age_days`**

**Available when?** Knowable at the decision moment because the age of the content is known before deciding whether to review or refresh the page.


In [11]:
feature_4 = df[["days_since_last_update"]].copy()

feature_4.head()

,days_since_last_update
0,20
1,25
2,20
3,22
4,14


**Feature 4: `days_since_last_update`**

**Available when?** Knowable at the decision moment because the number of days since the page was last updated is known before making the refresh decision.


In [12]:
feature_5 = df[["word_count"]].copy()

feature_5.head()

,word_count
0,3221.0
1,2481.0
2,3515.0
3,NaN
4,2803.0


**Feature 5: `word_count`**

**Available when?** Knowable at the decision moment because the page's word count is already known before deciding whether to review or refresh the content.


In [13]:
# Build the five-feature frame

features = [
    "search_volume",
    "competition",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

feature_df = df[features].copy()

print("Feature frame shape:", feature_df.shape)
print("\nFeatures:")
print(feature_df.columns.tolist())

feature_df.head()

Feature frame shape: (30000, 5)

Features:
['search_volume', 'competition', 'content_age_days', 'days_since_last_update', 'word_count']


,search_volume,competition,content_age_days,days_since_last_update,word_count
0,10.0,0.67,187,20,3221.0
1,90.0,0.01,445,25,2481.0
2,0.0,0.00,141,20,3515.0
3,10.0,0.00,463,22,NaN
4,0.0,0.00,263,14,2803.0


In [14]:
# Deliberate leakage experiment
# We intentionally include the label-derived field.

leaky_features = [
    "search_volume",
    "competition",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction"   # DELIBERATE LEAK
]

leaky_df = df[leaky_features].copy()

print("Leaky feature frame shape:", leaky_df.shape)
print("\nLeaky features:")
print(leaky_df.columns.tolist())

leaky_df.head()

Leaky feature frame shape: (30000, 6)

Leaky features:
['search_volume', 'competition', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction']


,search_volume,competition,content_age_days,days_since_last_update,word_count,trend_direction
0,10.0,0.67,187,20,3221.0,down
1,90.0,0.01,445,25,2481.0,down
2,0.0,0.00,141,20,3515.0,down
3,10.0,0.00,463,22,NaN,stable
4,0.0,0.00,263,14,2803.0,down


In [15]:
# Check the deliberate leakage
# trend_direction is directly used to create the target label.

leak_check = pd.crosstab(
    df["trend_direction"],
    df["trend_direction"].eq("down")
)

print(leak_check)

trend_direction  False  True 
trend_direction              
down                 0  16262
flat              1152      0
new               2236      0
stable            5962      0
up                4388      0


In [16]:
# Remove the leaked label-derived field
# Keep only the five honest features.

feature_df = df[
    [
        "search_volume",
        "competition",
        "content_age_days",
        "days_since_last_update",
        "word_count"
    ]
].copy()

print("Final feature frame shape:", feature_df.shape)
print("\nFinal features:")
print(feature_df.columns.tolist())

Final feature frame shape: (30000, 5)

Final features:
['search_volume', 'competition', 'content_age_days', 'days_since_last_update', 'word_count']


### Leakage lesson

I deliberately included `trend_direction` in the feature set. The check showed that it directly determines the target label: every `down` row matched the positive label and every other trend category matched the negative label.

This would make a model appear unrealistically accurate because it is being given information that is part of the outcome itself. Therefore, `trend_direction` is removed from the final feature frame.

The honest feature set contains only five fields:

* `search_volume`
* `competition`
* `content_age_days`
* `days_since_last_update`
* `word_count`

This is a deliberate check for label leakage, not a valid modeling feature.


## 4. Limitation

A limitation of this slice is that the dataset does not contain an explicit date or month field. Therefore, the exact calendar date range of each observation cannot be verified directly from this CSV. The available 30-day and 90-day performance fields provide relative recent-performance windows, but they do not replace an explicit observation date.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.